# Data Preparation for example_03d – Germany 2025

Pulls data from **ENTSO-E Transparency Platform API** and writes:
1. `availability_df.csv` – hourly availability factors (actual gen / monthly installed capacity, clipped [0,1])
2. `demand_df.csv` – hourly residual demand corrected for pumped hydro & batteries
3. `fuel_prices_df.csv` – hourly fuel prices (gas seasonal, CO2 from API if available)

**Requires:** ENTSO-E API key → transparency.entsoe.eu → My Account Settings → Web API Security Token

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from entsoe import EntsoePandasClient
from pathlib import Path

API_KEY = '28cf56f5-4b94-4270-b7df-007c082ed62e'  # <-- paste your key
COUNTRY = 'DE_LU'
START   = pd.Timestamp('2025-01-01', tz='Europe/Berlin')
END     = pd.Timestamp('2026-01-01', tz='Europe/Berlin')
OUT_DIR = Path('.')

client = EntsoePandasClient(api_key=API_KEY)
print('Client ready.')

In [ ]:
gen_raw = client.query_generation(COUNTRY, start=START, end=END, psr_type=None)
print('Raw shape:', gen_raw.shape)
gen_raw.head(2)

In [ ]:
# Resample to 1h, aggregate by technology, drop NaN columns
gen_h = gen_raw.resample('1h').mean()
gen_agg = gen_h.groupby(level=0).sum()

# ENTSO-E type strings → ASSUME model column names
TYPE_MAP_GEN = {
    'Wind Onshore':                    'Wind Onshore',
    'Wind Offshore':                   'Wind Offshore',
    'Solar':                           'Solar',
    'Hydro Run-of-river and poundage': 'Hydro',
    'Biomass':                         'Biomass',
    'Hydro Pumped Storage':            '_PumpedHydro_gen',
}
gen_mapped = gen_agg.rename(columns={k: v for k, v in TYPE_MAP_GEN.items() if k in gen_agg.columns})

RES_COLS = ['Wind Onshore', 'Wind Offshore', 'Solar', 'Hydro', 'Biomass']
missing  = [c for c in RES_COLS if c not in gen_mapped.columns]
if missing:
    print(f'WARNING missing in API response (will fill 0): {missing}')
    print('Available:', gen_agg.columns.tolist())

## 2 – Installed Capacity (monthly, for availability scaling)

Initially I wanted to take this from the historical monthly installed capacity. However I do not vary the capacity monthly so I should scale it with the installed capacity at the end of the year to ensure the infeed in the beginning of the year is not overestimated.

In [ ]:
# cap_raw: DataFrame with 'name' and 'max_power' from powerplant_units.csv
# Convert to a constant hourly DataFrame indexed like gen_h
cap_totals = cap_raw.set_index('name')['max_power']  # Series keyed by tech name

cap_h = pd.DataFrame(
    {col: cap_totals.get(col, np.nan) for col in RES_COLS},
    index=gen_h.index,
)
print(cap_h[RES_COLS].describe().loc[['min', 'max', 'mean']].round(0))

In [ ]:
avail = pd.DataFrame(index=gen_h.index)
for col in RES_COLS:
    gen_col = gen_h[col].fillna(0) if col in gen_h.columns else pd.Series(0.0, index=gen_h.index)
    if col in cap_h.columns and cap_h[col].max() > 0:
        avail[col] = (gen_col / cap_h[col]).clip(0, 1)
    else:
        mx = gen_col.max()
        avail[col] = (gen_col / mx).clip(0, 1) if mx > 0 else 0.0
        print(f'  {col}: normalised by yearly max (no capacity data)')
avail = avail.fillna(0.0)
print(avail.describe().round(3))

In [ ]:
# Strip timezone to naive UTC for ASSUME compatibility
if avail.index.tz is not None:
    avail.index = avail.index.tz_convert('UTC').tz_localize(None)
avail.index.name = 'datetime'
avail = avail.round(3)
avail.to_csv(OUT_DIR / 'availability_df.csv')
print('Written availability_df.csv  shape:', avail.shape)

In [ ]:
load_raw = client.query_load(COUNTRY, start=START, end=END)
# query_load may return a DataFrame or a Series depending on entsoe-py version
# .squeeze() converts a single-column DataFrame to a Series, leaves a Series unchanged
load_h   = load_raw.resample('1h').mean().squeeze()
assert isinstance(load_h, pd.Series), f'Expected Series, got {type(load_h)}'
print(f'Load: min={load_h.min():.0f} MW, max={load_h.max():.0f} MW, mean={load_h.mean():.0f} MW')

In [ ]:
# Pumped hydro: subtract generation, add consumption from net exchange
ph_gen = (gen_h['_PumpedHydro_gen'].fillna(0) 
          if '_PumpedHydro_gen' in gen_h.columns else pd.Series(0.0, index=gen_h.index))
# Net battery: from scheduled exchanges (simplified proxy)
net_battery = pd.Series(0.0, index=gen_h.index)  # Placeholder; refine with battery data if available

load_aligned = load_h.reindex(gen_h.index, method='nearest')
demand_corr  = (load_aligned - ph_gen - net_battery).clip(lower=0)
demand_corr.name = 'demand_EOM'
print(f'Corrected demand: min={demand_corr.min():.0f}, max={demand_corr.max():.0f}, mean={demand_corr.mean():.0f} MW')

# also plot orginal load for comparison
load_aligned.plot(figsize=(14, 3), lw=0.5, title='Original Load vs Corrected Demand [MW]', label='Original Load')
demand_corr.plot(figsize=(14, 3), lw=0.5, title='Original Load vs Corrected Demand [MW]', label='Corrected Demand')
plt.legend()    
plt.tight_layout()
plt.show()

## 3 – Demand DataFrame Export

In [ ]:
demand_df = pd.DataFrame({'demand': demand_corr})
if demand_df.index.tz is not None:
    demand_df.index = demand_df.index.tz_convert('UTC').tz_localize(None)
demand_df.index.name = 'datetime'
demand_df = demand_df.round(2)
demand_df.to_csv(OUT_DIR / 'demand_df.csv')
print('Written demand_df.csv  shape:', demand_df.shape)
demand_df.head()

## 4 – Scheduled Commercial Exchanges (cross-border net flows DE_LU)

In [ ]:
# Define DE_LU neighbors and their price codes
DE_NEIGHBOURS = {
    'AT': 'Austria', 'CZ': 'Czechia', 'DK': 'Denmark', 'FR': 'France',
    'NL': 'Netherlands', 'PL': 'Poland', 'CH': 'Switzerland'
}

# Query scheduled exchanges for each neighbor
exchanges_raw = pd.DataFrame(index=gen_h.index)
nb_prices = {}  # neighbor prices for weighted import price

for nb_code in DE_NEIGHBOURS.keys():
    try:
        ex = client.query_scheduled_exchanges(f'DE_LU', f'DE_LU-{nb_code}', start=START, end=END)
        ex_h = ex.resample('1h').mean()
        ex_h_pos = ex_h.clip(lower=0)  # imports (positive)
        ex_h_neg = ex_h.clip(upper=0).abs()  # exports (negative → take abs)
        
        col_prefix = DE_NEIGHBOURS[nb_code]
        exchanges_raw[f'{col_prefix}_import'] = ex_h_pos.reindex(gen_h.index, method='ffill').fillna(0)
        exchanges_raw[f'{col_prefix}_export'] = ex_h_neg.reindex(gen_h.index, method='ffill').fillna(0)
        
        # Also query day-ahead prices for that neighbor
        try:
            nb_da = client.query_day_ahead_prices(f'DE_LU-{nb_code}', start=START, end=END)
            nb_prices[nb_code] = nb_da.resample('1h').mean()
        except:
            nb_prices[nb_code] = pd.Series(0.0, index=gen_h.index)
    except Exception as e:
        print(f'Skipping {nb_code}: {e}')

print(f'Exchanges shape: {exchanges_raw.shape}')
print(f'Columns: {exchanges_raw.columns.tolist()}')

In [ ]:
exchanges_df = exchanges_raw.copy()
if exchanges_df.index.tz is not None:
    exchanges_df.index = exchanges_df.index.tz_convert('UTC').tz_localize(None)
exchanges_df.index.name = 'datetime'
exchanges_df = exchanges_df.round(2)
exchanges_df.to_csv(OUT_DIR / 'exchanges_df.csv')
print('Written exchanges_df.csv  shape:', exchanges_df.shape)

# Also write exchange_units.csv if not present
units_path = OUT_DIR / 'exchange_units.csv'
if not units_path.exists():
    units_path.write_text(
        'name,bidding_EOM,price_import,price_export,unit_operator\n'
        'exchange,exchange_energy_naive,0,2999,Exchanges Operator\n'
    )
    print('Written exchange_units.csv')

exchanges_df.head(3)

In [ ]:
# ── Volume-weighted average import price ──────────────────────────────────────
weighted_num = {}

for nb_code, p_h in nb_prices.items():
    col_prefix  = DE_NEIGHBOURS[nb_code]
    import_col  = f'{col_prefix}_import'
    if import_col not in exchanges_df.columns:
        continue
    flow = exchanges_df[import_col].clip(lower=0).fillna(0)
    price = p_h.reindex(flow.index, method='ffill').fillna(0)
    weighted_num[nb_code] = sum(flow * price) / sum(flow) if flow.sum() > 0 else 0

print(weighted_num)

## 5 – Fuel Prices 2025

ENTSO-E has no commodity prices. Estimates below; replace with Bloomberg/EEX/ICE time-series if available.

| Fuel | Basis | EUR/MWh_th or EUR/t | Source |
|------|-------|---------------------|--------|
| Natural gas | TTF 2025 forward + seasonal (+/-4) | 33–41 |  |
| Hard coal | EU ~125 EUR/t | 15.63 |  https://www.destatis.de/DE/Themen/Branchen-Unternehmen/Energie/Verwendung/Tabellen/einfuhr-steinkohle-jaehrlich.html |
| CO2 | EU ETS Phase 4 forward | 73 |
| Uranium | Long-term contract | 0.9 |
| Lignite | Domestic contract | 1.8 |
| Biomass | Wood pellets | 20.0 |
| Oil | Brent ~75 USD/bbl | 45.0 |

In [ ]:
co2_series = None
try:
    co2_raw    = client.query_carbon_pricing(COUNTRY, start=START, end=END)
    co2_series = co2_raw.resample('1h').ffill().reindex(gen_h.index, method='ffill')
    print(f'CO2 from ENTSO-E: mean={co2_series.mean():.1f} EUR/t')
except Exception as e:
    print(f'CO2 endpoint not available ({e}) – using flat 73 EUR/t')

In [ ]:
# ── Kaggle dataset: gas + CO2 prices for 2025 ─────────────────────────────────
# Source: kaggle.com/datasets/williamdennis/de-lu-electricity-market
#
# carbon_price_usd  → CARB.L  WisdomTree Carbon ETC (LSE, Yahoo Finance)
#                     USD-denominated EUA futures proxy, daily → hourly ffill
#                     Tracks EU ETS front-month futures; small tracking error
#                     vs. EEX spot due to roll cost & USD/EUR FX
#
# gas_price_usd     → TTF front-month in USD, daily → hourly ffill
import kagglehub, requests

kaggle_path = kagglehub.dataset_download("williamdennis/de-lu-electricity-market")

kg = pd.read_csv(
    f'{kaggle_path}/dataset_de_lu.csv',
    usecols=['timestamp_utc', 'carbon_price_usd', 'gas_price_usd'],
    parse_dates=['timestamp_utc'],
)
kg['timestamp_utc'] = pd.to_datetime(kg['timestamp_utc'], utc=True)
kg = kg.set_index('timestamp_utc').sort_index()
kg_2025 = kg.loc['2025-01-01':'2025-12-31'].resample('1h').mean()

# ── Daily USD→EUR rate from ECB via Frankfurter API (no auth needed) ──────────
resp = requests.get(
    'https://api.frankfurter.app/2025-01-01..2025-12-31',
    params={'from': 'USD', 'to': 'EUR'},
    timeout=30,
)
resp.raise_for_status()
fx_daily = pd.Series(
    {pd.Timestamp(d, tz='UTC'): v['EUR'] for d, v in resp.json()['rates'].items()}
).sort_index()
# Forward-fill to hourly (weekends/holidays carry last business day rate)
fx_h = fx_daily.resample('1h').ffill().reindex(kg_2025.index, method='ffill')
print(f'EUR/USD rate 2025:  mean={fx_h.mean():.4f}  min={fx_h.min():.4f}  max={fx_h.max():.4f}')

# ── Convert USD → EUR using daily rate ────────────────────────────────────────
kg_2025['carbon_eur_t'] = (kg_2025['carbon_price_usd'] * fx_h).round(4)
kg_2025['gas_eur_mwh']  = (kg_2025['gas_price_usd']    * fx_h).round(4)

# rescale carbon price to 73 €/t to match general german carbon level
# https://www.bayern-innovativ.de/en/emagazine/detail/germany-increases-revenue-from-co2-trading-in-2025
kg_2025['carbon_eur_t'] = kg_2025['carbon_eur_t'] * (73 / kg_2025['carbon_eur_t'].mean()).round(4)

print(f'Kaggle 2025 gas:  mean={kg_2025["gas_eur_mwh"].mean():.2f} EUR/MWh  '
      f'min={kg_2025["gas_eur_mwh"].min():.2f}  max={kg_2025["gas_eur_mwh"].max():.2f}')
print(f'Kaggle 2025 CO2 (CARB.L→EUA proxy):  mean={kg_2025["carbon_eur_t"].mean():.2f} EUR/t  '
      f'min={kg_2025["carbon_eur_t"].min():.2f}  max={kg_2025["carbon_eur_t"].max():.2f}')

# ── Remaining fuels: scale 2019 time series to 2025 target means ──────────────
# Hard coal: 120.3 €/t with LHV=27.710 MJ/t → 120.3 / (27.710/3.6) = 15.63 €/MWh
FUEL_TARGETS_2025 = {
    'uranium':   0.9,
    'lignite':   1.8,
    'hard coal': 15.63,  # Updated from Statista 2025: 120.3 EUR/t / (27.710 MJ/t / 3.6)
    'oil':       45.0,
    'biomass':   20.0,
}

prices_2019 = pd.read_csv(OUT_DIR / 'fuel_prices_df.csv', index_col=0, parse_dates=True)
prices_2019_h = prices_2019.resample('1h').mean()

fuel_df = pd.DataFrame(index=gen_h.index)  # timezone-aware 2025 index

for col, target in FUEL_TARGETS_2025.items():
    if col in prices_2019_h.columns:
        mean_2019 = prices_2019_h[col].mean()
        scale = target / mean_2019 if mean_2019 != 0 else 1.0
        vals = prices_2019_h[col].values[:len(fuel_df)]
        fuel_df[col] = (vals * scale).round(4)
        print(f'  {col}: mean_2019={mean_2019:.3f}  scale={scale:.3f}  → mean_2025={fuel_df[col].mean():.3f}')

In [ ]:
if fuel_df.index.tz is not None:
    fuel_df.index = fuel_df.index.tz_convert('UTC').tz_localize(None)
fuel_df.index.name = 'datetime'
fuel_df.to_csv('fuel_prices_df.csv')
print('Written fuel_prices_df.csv  shape:', fuel_df.shape)
fuel_df.head()

In [ ]:
for name, df in [('availability_df', avail), ('demand_df', demand_df), ('fuel_prices_df', fuel_df)]:
    na = df.isna().sum().sum()
    print(f'{name}: {len(df)} rows, {na} NaNs, cols={df.columns.tolist()}')

# 2025 is not a leap year → 8760 hourly rows
assert len(avail)     == 8760, f'avail {len(avail)} rows != 8760'
assert len(demand_df) == 8760, f'demand {len(demand_df)} rows != 8760'
assert len(fuel_df)   == 8760, f'fuel_prices {len(fuel_df)} rows != 8760'
print('All row-count assertions passed.')

## 6 – Power Plant List

Die Kraftwerksliste wurde gegen die **Bundesnetzagentur Kraftwerksliste** (MaStR-Export, Spalte *„Kraftwerksstatus der Einheit"*) abgeglichen.

### Entfernte Einheiten

**Eindeutig nicht „In Betrieb
7
8
4
4
3
1
,
5
7
,
13
,
,
,
1
2018
,
2024
,
,
5
,
,
,

- VOERDE A, IRSCHING 3, LUENEN 2/7, ZOLLING LEININGER 5 → individuell geprüft
- Noch mal alle Braunkohle Kraftwerke weil da fast immer Teil der Blöcke abgeschaltet wurden

### Quelle
`Kraftwerksliste (1).xlsx` – BNetzA MaStR-Export, Abgleich über Namens-Fuzzy-Matching + Nettonennleistung

## 7 – Pre-Check Results

Run the scenario with the heuristic to check if we are somewhat in the correct realm of prices.

In [1]:
from sqlalchemy import create_engine
import examples

example = "case_study_2025"
db_uri = "sqlite:///./examples/local_db/assume_db_8459abd.db"

inputs_dir = "examples/inputs"

scenario = "example_03d"
study_case = "base_heu"

# Set up the database connection
db = create_engine(db_uri)
print(f'Connected to {db_uri}')

ModuleNotFoundError: No module named 'examples'

In [ ]:
start_plot = pd.Timestamp('2025-05-26 00:00:00', tz='UTC')
end_plot = pd.Timestamp('2025-06-02 00:00:00', tz='UTC')

# ── ENTSO-E Day-Ahead Prices DE_LU 2025 ───────────────────────────────────────
da_raw = client.query_day_ahead_prices(COUNTRY, start=start_plot, end=end_plot)
da_h   = da_raw.resample('1h').mean().squeeze()
if da_h.index.tz is not None:
    da_h.index = da_h.index.tz_convert('UTC').tz_localize(None)
print(f'Day-ahead prices: {len(da_h)} hours  '
      f'mean={da_h.mean():.1f}  min={da_h.min():.1f}  max={da_h.max():.1f} EUR/MWh')

In [ ]:
# Load simulation results from database
try:
    market_meta_df = pd.read_sql_table('market_meta_df', db)
    print(f'Loaded market_meta_df: {len(market_meta_df)} rows')
except Exception as e:
    print(f'No market data available: {e}')
    market_meta_df = pd.DataFrame()

try:
    dispatch_df = pd.read_sql_table('dispatch_df', db)
    print(f'Loaded dispatch_df: {len(dispatch_df)} rows')
except Exception as e:
    print(f'No dispatch data: {e}')
    dispatch_df = pd.DataFrame()

try:
    pp_meta_df = pd.read_sql_table('pp_meta_df', db)
    print(f'Loaded pp_meta_df: {len(pp_meta_df)} rows')
except Exception as e:
    print(f'No powerplant metadata: {e}')
    pp_meta_df = pd.DataFrame()

In [ ]:
# ── Price Duration Curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# ENTSO-E historical
da_sorted = da_h.dropna().sort_values(ascending=False).reset_index(drop=True)
x_da = np.linspace(0, 100, len(da_sorted))
ax.plot(x_da, da_sorted.values, lw=1.5, color='steelblue', label=f'ENTSO-E Day-Ahead (mean={da_h.mean():.0f} €/MWh)')

# Simulated
if not market_meta_df.empty:
    sorted_prices = np.sort(market_meta_df["price"].dropna().values)[::-1]
    if len(sorted_prices) > 0:
        x_sim = np.linspace(0, 100, len(sorted_prices))
        ax.plot(x_sim, sorted_prices, lw=1.5, color='tomato',
                label=f'ASSUME simulated (mean={market_meta_df["price"].mean():.0f} €/MWh)')
else:
    ax.text(50, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else 0,
            'Simulated prices not yet available', ha='center', color='grey', fontsize=9)

ax.set_xlabel('% of hours')
ax.set_ylabel('Price [EUR/MWh]')
ax.set_title('Price Duration Curve – DE_LU 2025')
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.legend()
ax.set_xlim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Summary stats
print(f'\n{"":25} {"ENTSO-E":>10} {"Simulated":>10}')
for label, fn in [('Mean', np.mean), ('Median', np.median),
                   ('P10', lambda x: np.percentile(x, 10)),
                   ('P90', lambda x: np.percentile(x, 90)),
                   ('Min', np.min), ('Max', np.max)]:
    sim_val = f'{fn(sorted_prices):>10.1f}' if not market_meta_df.empty else f'{"n/a":>10}'
    print(f'{label:25} {fn(da_h.dropna()):>10.1f} {sim_val}')